#Get data from youtube

In [7]:
!pip -q install -r requirements.txt


In [9]:
import os, glob, cv2, math, json
import numpy as np
import pandas as pd
from tqdm import tqdm
from fer import FER
from pathlib import Path
import yt_dlp
import easyocr
from rapidfuzz import fuzz
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')


VIDEO_ID   = "hhLOQ0S-uJo"
VIDEO_URL  = f"https://www.youtube.com/watch?v={VIDEO_ID}"
BASE_DIR   = "/content/emo_sign_single"
VIDEOS_DIR = f"{BASE_DIR}/videos"
OUT_DIR    = f"{BASE_DIR}/outputs"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(VIDEOS_DIR, exist_ok=True)
os.makedirs(f"{BASE_DIR}/outputs", exist_ok=True)

TRANS_OUT_JSON = f"{OUT_DIR}/{VIDEO_ID}_transcripts_ocr.json"


TRANS_BASE_JSON = f"{BASE_DIR}/outputs/{VIDEO_ID}_transcripts_ocr.json"
SENTS_CSV  = f"{BASE_DIR}/outputs/{VIDEO_ID}_sentiment_vader_ocr.csv"

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [10]:

from getpass import getpass
API_KEY = getpass("Paste your YouTube API key (input hidden): ")


Paste your YouTube API key (input hidden): ··········


#Download data

In [11]:


ydl_opts = {
    "outtmpl": f"{VIDEOS_DIR}/%(id)s.%(ext)s",
    "format": "mp4[height<=480]/mp4/best[ext=mp4]/best",
    "noplaylist": True,
    "ignoreerrors": False,
    "retries": 10,
    "fragment_retries": 10,
    "concurrent_fragment_downloads": 5,
    "geo_bypass": True,
    "quiet": False,
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(VIDEO_URL, download=True)

print("yt-dlp returned id/ext:", info.get("id"), info.get("ext"))
candidates = glob.glob(f"{VIDEOS_DIR}/{info.get('id') or VIDEO_ID}.*")
print("Downloaded files:", candidates)
src = candidates[0]
print("Using src:", src)


[youtube] Extracting URL: https://www.youtube.com/watch?v=hhLOQ0S-uJo
[youtube] hhLOQ0S-uJo: Downloading webpage
[youtube] hhLOQ0S-uJo: Downloading tv simply player API JSON
[youtube] hhLOQ0S-uJo: Downloading tv client config
[youtube] hhLOQ0S-uJo: Downloading tv player API JSON
[youtube] hhLOQ0S-uJo: Downloading player 0004de42-main
[info] hhLOQ0S-uJo: Downloading 1 format(s): 18
[download] Destination: /content/emo_sign_single/videos/hhLOQ0S-uJo.mp4
[download] 100% of    3.79MiB in 00:00:00 at 7.71MiB/s   
yt-dlp returned id/ext: hhLOQ0S-uJo mp4
Downloaded files: ['/content/emo_sign_single/videos/hhLOQ0S-uJo.mp4']
Using src: /content/emo_sign_single/videos/hhLOQ0S-uJo.mp4


#Emotional detection

In [12]:
VIDEO_PATH = glob.glob(f"{BASE_DIR}/videos/{VIDEO_ID}.*")[0]
SAMPLE_FPS = 3.0
detector = FER(mtcnn=True)
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = frame_count / max(fps, 1e-6)
step = max(1, int(round(fps / SAMPLE_FPS)))
records = []
idx = -1
pbar = tqdm(total=frame_count//step + 1, desc="Emotion frames")

while True:
    ok, frame = cap.read()
    if not ok: break
    idx += 1
    if idx % step != 0:
        continue
    t = idx / fps
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    dets = detector.detect_emotions(rgb)
    if dets:
        best = max(dets, key=lambda d: max(d["emotions"].values()))
        probs = best["emotions"]
        top_emotion = max(probs, key=probs.get)
        rec = {"time_sec": round(t,3), "top_emotion": top_emotion, **{f"p_{k}": float(v) for k,v in probs.items()}}
    else:
        rec = {"time_sec": round(t,3), "top_emotion": "no_face"}
    records.append(rec)
    pbar.update(1)
cap.release()
pbar.close()

frame_csv = f"{BASE_DIR}/outputs/{VIDEO_ID}_frame_emotions.csv"
df = pd.DataFrame(records)
df.to_csv(frame_csv, index=False)
print("Per-frame emotions CSV:", frame_csv)

emo_cols = [c for c in df.columns if c.startswith("p_")]
summary = {}
if len(df):
    mode_series = df.loc[df["top_emotion"]!="no_face","top_emotion"]
    dominant = mode_series.mode().iloc[0] if len(mode_series) else "no_face"
    mean_probs = df.loc[df["top_emotion"]!="no_face", emo_cols].mean().to_dict() if len(mode_series) else {}
    summary = {
        "video_id": VIDEO_ID,
        "duration_sec": duration,
        "frames_sampled": int(len(df)),
        "dominant_emotion": dominant,
        "mean_emotions": mean_probs
    }

summary_json = f"{BASE_DIR}/outputs/{VIDEO_ID}_video_emotion_summary.json"
with open(summary_json, "w") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print("Video summary JSON:", summary_json)
summary


Emotion frames:   0%|          | 1/298 [00:00<00:33,  9.00it/s]WARNING:py.warnings:/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['input_1']
Received: inputs=Tensor(shape=(2, 64, 64))
  warnings.warn(msg)

Emotion frames:  10%|█         | 30/298 [00:11<02:17,  1.95it/s]WARNING:py.warnings:/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['input_1']
Received: inputs=Tensor(shape=(3, 64, 64))
  warnings.warn(msg)

Emotion frames:  20%|██        | 60/298 [00:22<01:18,  3.04it/s]WARNING:py.warnings:/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['input_1']
Received: inputs=Tensor(shape=(1, 64, 64))
  warnings.warn(msg)

Emotion frames:  53%|█████▎  

Per-frame emotions CSV: /content/emo_sign_single/outputs/hhLOQ0S-uJo_frame_emotions.csv
Video summary JSON: /content/emo_sign_single/outputs/hhLOQ0S-uJo_video_emotion_summary.json


{'video_id': 'hhLOQ0S-uJo',
 'duration_sec': 99.26583333333333,
 'frames_sampled': 298,
 'dominant_emotion': 'neutral',
 'mean_emotions': {'p_angry': 0.04893470790378007,
  'p_disgust': 0.0019243986254295535,
  'p_fear': 0.049484536082474224,
  'p_happy': 0.1234364261168385,
  'p_sad': 0.09549828178694159,
  'p_surprise': 0.010206185567010308,
  'p_neutral': 0.6692783505154639}}

#Segmentation + Sentiment Analysis

In [13]:
ROI_BOTTOM_FRAC = 0.55    # scan bottom 35% of the frame (typical subtitle area)
CONF_MIN         = 0.45   # min OCR confidence per word
MERGE_SIM        = 78     # min similarity (0-100) to treat consecutive texts as same subtitle
MAX_GAP_SEC      = 1.2   # allow brief gaps (subtitle flicker) when merging
JOIN_DETECTIONS  = True   # if multiple boxes per frame, join texts with spaces

In [14]:
use_gpu = Path("/proc/driver/nvidia/version").exists()
reader = easyocr.Reader(['en'], gpu=use_gpu, verbose=False)

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
step = max(1, int(round(fps / SAMPLE_FPS)))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = frame_count / max(fps, 1e-6)

samples = []

idx = -1
while True:
    ok, frame = cap.read()
    if not ok:
        break
    idx += 1
    if idx % step != 0:
        continue
    t = idx / fps
    H, W = frame.shape[:2]
    y0 = int(H * (1.0 - ROI_BOTTOM_FRAC))
    roi = frame[y0:H, 0:W]
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray = cv2.bilateralFilter(gray, 5, 40, 40)
    roi_proc = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    detections = reader.readtext(roi_proc, detail=1, paragraph=False, min_size=10,
                                 text_threshold=0.6, low_text=0.3, link_threshold=0.5)
    texts = [d[1].strip() for d in detections if d[2] >= CONF_MIN and d[1].strip()]
    if not texts:
        continue
    text = " ".join(texts) if JOIN_DETECTIONS else max(texts, key=len)
    text = " ".join(text.split())
    if len(text) < 2:
        continue
    samples.append((t, text))
cap.release()

segments = []
if samples:
    current_text = samples[0][1]
    start_t = last_t = samples[0][0]

    for t, txt in samples[1:]:
        sim = fuzz.token_set_ratio(current_text, txt)
        if sim >= MERGE_SIM and (t - last_t) <= MAX_GAP_SEC:
            last_t = t
            if len(txt) > len(current_text):
                current_text = txt
        else:
            end_t = last_t + (1.0 / SAMPLE_FPS)
            segments.append({"text": current_text, "start": float(start_t), "duration": float(max(0.1, end_t - start_t))})
            current_text = txt
            start_t = last_t = t

    end_t = last_t + (1.0 / SAMPLE_FPS)
    segments.append({"text": current_text, "start": float(start_t), "duration": float(max(0.1, end_t - start_t))})

with open(TRANS_OUT_JSON, "w", encoding="utf-8") as f:
    json.dump({VIDEO_ID: segments}, f, ensure_ascii=False, indent=2)

print(f"OCR transcript segments: {len(segments)}")
print("Saved:", TRANS_OUT_JSON)
pd.DataFrame(segments).head(10)


  warnings.warn(warn_msg)



OCR transcript segments: 14
Saved: /content/emo_sign_single/outputs/hhLOQ0S-uJo_transcripts_ocr.json


,text,start,duration
0,Chao mung cac ban den voi Ban tin Xa hoi:,1.001000,2.669000
1,That su' rat xuc dong!,3.670333,3.002667
2,"Gan day; qua Hoi chu thap do, chinh phu da pha...",6.673333,4.003667
3,"ung ho nhan dan Cuba voi chu de ""65 nam nghia ...",10.677333,12.345333
4,65.000.0oo.000 Dat muc tieu VND Trung Tnarn ph...,23.023000,8.007667
5,65.000.000.000 Dat muc tieu VND Trung Thanh ph...,31.031000,7.006667
6,niennangcamauquainnh Nguoi dan Viet Nam luon g...,38.038000,4.337333
7,"nien dargcamaucuainnn Nam 1966, lanh tu Fidel ...",42.375667,8.341333
8,"chien tranh, Cuba da ho trd thuoc men, luong t...",50.717333,9.342333
9,Nhan dan Viet Nam luon khac ghi va danh cho nh...,60.060000,10.677000


In [22]:
with open(TRANS_BASE_JSON, "r", encoding="utf-8") as f:
    tx = json.load(f)
segments = tx.get(VIDEO_ID, []) or []

sia = SentimentIntensityAnalyzer()

def vader_label(c):
    return "unavailable" if c is None else ("positive" if c>=0.05 else ("negative" if c<=-0.05 else "neutral"))
def intensity(c):
    if c is None: return "unavailable"
    m=abs(c); return "low" if m<0.25 else ("medium" if m<0.55 else "high")

rows=[]
for s in segments:
    sc = sia.polarity_scores(s["text"])
    rows.append({
        "level": "segment",
        "start_sec": s["start"],
        "duration_sec": s["duration"],
        "text": s["text"],
        "compound": sc["compound"],
        "positive": sc["pos"],
        "neutral": sc["neu"],
        "negative": sc["neg"],
        "sentiment_label": vader_label(sc["compound"]),
        "sentiment_intensity": intensity(sc["compound"]),
    })

if segments:
    full_text = " ".join(s["text"] for s in segments).strip()
    sc = sia.polarity_scores(full_text)
    rows.insert(0, {
        "level": "overall",
        "start_sec": 0.0,
        "duration_sec": sum(s["duration"] for s in segments),
        "text": full_text[:5000],
        "compound": sc["compound"],
        "positive": sc["pos"],
        "neutral": sc["neu"],
        "negative": sc["neg"],
        "sentiment_label": vader_label(sc["compound"]),
        "sentiment_intensity": intensity(sc["compound"]),
    })


sent_df = pd.DataFrame(rows)
sent_df.to_csv(SENTS_CSV, index=False)
print("Saved:", SENTS_CSV)
sent_df.head(5)


Saved: /content/emo_sign_single/outputs/hhLOQ0S-uJo_sentiment_vader_ocr.csv


,level,start_sec,duration_sec,text,compound,positive,neutral,negative,sentiment_label,sentiment_intensity
0,overall,0.000000,97.759667,Chao mung cac ban den voi Ban tin Xa hoi: That...,-0.9098,0.035,0.893,0.072,negative,high
1,segment,1.001000,2.669000,Chao mung cac ban den voi Ban tin Xa hoi:,-0.8020,0.000,0.526,0.474,negative,high
2,segment,3.670333,3.002667,That su' rat xuc dong!,0.0000,0.000,1.000,0.000,neutral,low
3,segment,6.673333,4.003667,"Gan day; qua Hoi chu thap do, chinh phu da pha...",0.0000,0.000,1.000,0.000,neutral,low
4,segment,10.677333,12.345333,"ung ho nhan dan Cuba voi chu de ""65 nam nghia ...",0.0000,0.000,1.000,0.000,neutral,low


In [23]:
sent_df

,level,start_sec,duration_sec,text,compound,positive,neutral,negative,sentiment_label,sentiment_intensity
0,overall,0.000000,97.759667,Chao mung cac ban den voi Ban tin Xa hoi: That...,-0.9098,0.035,0.893,0.072,negative,high
1,segment,1.001000,2.669000,Chao mung cac ban den voi Ban tin Xa hoi:,-0.8020,0.000,0.526,0.474,negative,high
2,segment,3.670333,3.002667,That su' rat xuc dong!,0.0000,0.000,1.000,0.000,neutral,low
3,segment,6.673333,4.003667,"Gan day; qua Hoi chu thap do, chinh phu da pha...",0.0000,0.000,1.000,0.000,neutral,low
4,segment,10.677333,12.345333,"ung ho nhan dan Cuba voi chu de ""65 nam nghia ...",0.0000,0.000,1.000,0.000,neutral,low
5,segment,23.023000,8.007667,65.000.0oo.000 Dat muc tieu VND Trung Tnarn ph...,0.3400,0.107,0.893,0.000,positive,medium
6,segment,31.031000,7.006667,65.000.000.000 Dat muc tieu VND Trung Thanh ph...,0.6124,0.217,0.783,0.000,positive,high
7,segment,38.038000,4.337333,niennangcamauquainnh Nguoi dan Viet Nam luon g...,0.0000,0.000,1.000,0.000,neutral,low
8,segment,42.375667,8.341333,"nien dargcamaucuainnn Nam 1966, lanh tu Fidel ...",0.0000,0.000,1.000,0.000,neutral,low
9,segment,50.717333,9.342333,"chien tranh, Cuba da ho trd thuoc men, luong t...",0.0000,0.000,1.000,0.000,neutral,low


#Qwen2.5 + AffectGPT

In [ ]:
# =========================
# AffectGPT-lite in Colab: Qwen2.5-VL + OCR (no audio)
# =========================

import os, glob, json, math, cv2, torch, re
import numpy as np, pandas as pd
from PIL import Image
from tqdm import tqdm

# ---------- Config ----------
VIDEO_ID   = "hhLOQ0S-uJo"
BASE_DIR   = "/content/emo_sign_single"
VIDEOS_DIR = f"{BASE_DIR}/videos"
OUT_DIR    = f"{BASE_DIR}/outputs_affectgpt_lite"
os.makedirs(VIDEOS_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

VIDEO_URL  = f"https://www.youtube.com/watch?v={VIDEO_ID}"
SAMPLE_FPS = 1.0                 # frames per second (increase for finer granularity)
MAX_FRAMES_PER_CALL = 8          # how many frames to send per Qwen prompt
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

# ---------- 1) Download the video (best available) ----------
import yt_dlp
video_path = None
outtmpl = f"{VIDEOS_DIR}/{VIDEO_ID}.%(ext)s"
ydl_opts = {
    "format": "bestvideo[ext=mp4]+bestaudio/best/best",
    "outtmpl": outtmpl,
    "quiet": True,
    "merge_output_format": "mp4"
}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([VIDEO_URL])

cands = glob.glob(f"{VIDEOS_DIR}/{VIDEO_ID}.*")
assert len(cands), "Failed to download video."
VIDEO_PATH = sorted(cands, key=len)[0]

# ---------- 2) Init models: Qwen2.5-VL + OCR ----------
from transformers import AutoProcessor, AutoModelForVision2Seq
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
# fp16 for speed if GPU is present
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device=="cuda" else None,
    device_map="auto",
    trust_remote_code=True
)

import easyocr
ocr_reader = easyocr.Reader(['en'])  # OCR for burned-in subs

def ocr_frame_text(bgr_img):
    res = ocr_reader.readtext(bgr_img)
    return " ".join([r[1] for r in res]).strip()

# ---------- 3) Sample frames ----------
cap = cv2.VideoCapture(VIDEO_PATH); assert cap.isOpened(), "Cannot open video."
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
duration = total / max(fps, 1e-6)
step = max(1, int(round(fps / SAMPLE_FPS)))

frames, metas = [], []
idx = -1
pbar = tqdm(total=total//step + 1, desc="Sampling frames")
while True:
    ok, frame = cap.read()
    if not ok: break
    idx += 1
    if idx % step != 0:
        continue
    t = idx / fps
    frames.append(frame)
    metas.append({"time_sec": round(float(t), 3)})
    pbar.update(1)
cap.release(); pbar.close()

# ---------- 4) Prompt Qwen for open-vocab emotions ----------
def normalize_labels(free_text):
    # You can tune this vocab as you like
    label_list = [
        "joy","happiness","sadness","anger","surprise","fear","disgust",
        "confusion","neutral","excitement","calm","frustration","contempt",
        "anxiety","pride","trust","hope","embarrassment","relief","boredom"
    ]
    found = []
    low = free_text.lower()
    for lab in label_list:
        if lab in low:
            found.append(lab)
    return sorted(set(found))

records = []
for i in tqdm(range(0, len(frames), MAX_FRAMES_PER_CALL), desc="AffectGPT-lite inference"):
    chunk = frames[i:i+MAX_FRAMES_PER_CALL]
    chunk_meta = metas[i:i+MAX_FRAMES_PER_CALL]
    if not chunk:
        continue

    # Convert to PIL and OCR the last frame for on-screen transcript cue
    pil_imgs = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in chunk]
    ocr_txt = ocr_frame_text(chunk[-1]) if len(chunk) else ""

    # Compose the prompt
    user_prompt = (
        "You are an expert in open-vocabulary multimodal emotion understanding. "
        "Given these frames from a sign-language video, list the perceived emotions "
        "(top 1–5 concise emotion words) and provide a one-sentence rationale based "
        "on facial expressions and body cues."
    )
    if ocr_txt:
        user_prompt += f'\nVisible on-screen text (OCR): "{ocr_txt}".'
    user_prompt += "\nReturn STRICT JSON with keys: emotions (list[str]), rationale (str)."

    messages = [
        {"role": "system", "content": "You are AffectGPT-like: do open-vocabulary multimodal emotion recognition."},
        {"role": "user", "content": [{"type":"text","text":user_prompt}] + \
                            [{"type":"image","image":img} for img in pil_imgs]}
    ]

    # Apply Qwen chat template + encode images
    inputs_txt = processor.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(device)
    inputs = processor(images=pil_imgs, text=inputs_txt, return_tensors="pt").to(device)

    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=256)
    out_text = processor.batch_decode(generated, skip_special_tokens=True)[0]

    # Extract JSON if possible; else keep raw
    js = {"emotions": [], "rationale": out_text.strip()}
    m = re.search(r'\{.*\}', out_text, re.S)
    if m:
        try:
            js = json.loads(m.group(0))
        except Exception:
            pass

    # Normalize labels using your controlled vocab
    labs = js.get("emotions", []) if isinstance(js.get("emotions", []), list) else []
    norm = normalize_labels(" ".join(labs) + " " + js.get("rationale",""))

    # Assign to each frame in the chunk
    for meta in chunk_meta:
        records.append({
            "time_sec": meta["time_sec"],
            "emotions_raw": labs,
            "emotions_norm": norm,
            "rationale": js.get("rationale",""),
            "ocr_text": ocr_txt
        })

df = pd.DataFrame(records).sort_values("time_sec").reset_index(drop=True)
csv_path = f"{OUT_DIR}/{VIDEO_ID}_affectgptlite.csv"
df.to_csv(csv_path, index=False)
print("Saved per-frame CSV:", csv_path)

# ---------- 5) Optional: text-only sentiment on OCR (per frame) ----------
# If you want a standard sentiment label (negative/neutral/positive) from OCR text:
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 >/dev/null 2>&1
!pip -q install transformers --upgrade >/dev/null 2>&1
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

sent_model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tok = AutoTokenizer.from_pretrained(sent_model_name)
smdl = AutoModelForSequenceClassification.from_pretrained(sent_model_name).to(device)

def senti_label(text):
    if not text or not text.strip():
        return {"sentiment_label":"unknown","sentiment_score":None}
    inputs = tok(text, truncation=True, max_length=128, return_tensors="pt").to(device)
    with torch.inference_mode():
        logits = smdl(**inputs).logits
        probs = F.softmax(logits, dim=-1).squeeze().tolist()
    idx2lab = ["negative","neutral","positive"]
    top_idx = int(np.argmax(probs))
    return {"sentiment_label": idx2lab[top_idx], "sentiment_score": float(probs[top_idx])}

sent_rows = []
for i, row in df.iterrows():
    s = senti_label(row["ocr_text"])
    sent_rows.append(s)
sent_df = pd.DataFrame(sent_rows)
df2 = pd.concat([df, sent_df], axis=1)

csv2 = f"{OUT_DIR}/{VIDEO_ID}_affectgptlite_with_sentiment.csv"
df2.to_csv(csv2, index=False)
print("Saved per-frame CSV (+sentiment):", csv2)

# ---------- 6) Aggregate summary ----------
from collections import Counter
agg = Counter()
for labs in df2["emotions_norm"]:
    for x in labs: agg[x]+=1

dominant = [k for k,_ in agg.most_common(5)]
summary = {
    "video_id": VIDEO_ID,
    "duration_sec": duration,
    "frames_sampled": len(df2),
    "top_emotions": dominant,
    "label_counts": dict(agg),
    "sentiment_counts": df2["sentiment_label"].value_counts(dropna=False).to_dict()
}
sum_path = f"{OUT_DIR}/{VIDEO_ID}_summary.json"
with open(sum_path, "w") as f: json.dump(summary, f, indent=2)
summary


ERROR: [youtube] hhLOQ0S-uJo: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


DownloadError: ERROR: [youtube] hhLOQ0S-uJo: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies

In [ ]:
!pip -q install transformers accelerate sentencepiece easyocr --upgrade


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 63.1 MB/s eta 0:00:00


In [ ]:
import re, json, torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForVision2Seq
import easyocr

# --------------------
# Qwen settings
# --------------------
MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"     # try 3B if OOM
MAX_FRAMES_PER_CALL = 2                        # frames sent per prompt
EMO = ["angry","disgust","fear","happy","sad","surprise","neutral"]
SAMPLE_FPS = 2

device = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device=="cuda" else None,
    device_map="auto",
    trust_remote_code=True
)
ocr_reader = easyocr.Reader(['en'])

def ocr_text_from_frame(bgr_img):
    """Simple OCR of the last frame in a chunk (optional cue)."""
    res = ocr_reader.readtext(bgr_img)
    return " ".join([r[1] for r in res]).strip()

def qwen_classify_chunk(frames_bgr):
    """Return (top_label, scores_dict) for a small chunk of frames."""
    if not frames_bgr:
        return "neutral", {k:0.0 for k in EMO}

    # Convert frames to PIL RGB
    pil_imgs = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in frames_bgr]

    # Optional on-screen text cue from last frame (burned-in subs)
    ocr_txt = ocr_text_from_frame(frames_bgr[-1])

    # Strict instruction: 7 labels only + probabilities that sum ~1
    prompt = (
        "Classify the overall facial emotion expressed in these video frames. "
        "Use exactly these labels: angry, disgust, fear, happy, sad, surprise, neutral. "
        "Return STRICT JSON with keys: "
        "'scores' (object mapping each of the 7 labels to a probability between 0 and 1 that sums to 1), "
        "and 'top' (string, the label with highest probability). "
        "Do not include any extra keys or commentary."
    )
    if ocr_txt:
        prompt += f'\nOn-screen text (for context): "{ocr_txt}"'

    messages = [
        {"role": "system", "content": "You are a precise visual emotion classifier."},
        {"role": "user", "content": [{"type":"text","text":prompt}] +
                                    [{"type":"image","image":img} for img in pil_imgs]}
    ]

    chat_text = processor.apply_chat_template(messages, add_generation_prompt=True)  # returns str
    inputs = processor(images=pil_imgs, text=chat_text, return_tensors="pt").to(device)

    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=220)
    out_text = processor.batch_decode(generated, skip_special_tokens=True)[0]

    # Parse STRICT JSON; be defensive
    scores = {k:0.0 for k in EMO}
    top = "neutral"
    m = re.search(r"\{.*\}", out_text, re.S)
    if m:
        try:
            js = json.loads(m.group(0))
            js_scores = js.get("scores", {})
            for k in EMO:
                if k in js_scores:
                    scores[k] = float(js_scores[k])
            ssum = sum(scores.values())
            if ssum > 0:
                scores = {k: v/ssum for k, v in scores.items()}  # normalize just in case
            top = js.get("top", top)
            if top not in EMO:
                top = max(scores, key=scores.get)
        except Exception:
            # keep defaults on parse errors
            pass

    return top, scores

# --------------------
# Video sampling (reuse your variables)
# --------------------
cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), "Cannot open video."

fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
duration = frame_count / max(fps, 1e-6)
step = max(1, int(round(fps / SAMPLE_FPS)))

records = []
idx = -1
pbar = tqdm(total=frame_count//step + 1, desc="Qwen emotion frames")

# small buffer so we send a few frames at once
buf_frames, buf_times = [], []

def flush_buffer():
    if not buf_frames:
        return
    top, scores = qwen_classify_chunk(buf_frames)
    for t in buf_times:
        rec = {"time_sec": round(t, 3), "top_emotion": top}
        rec.update({f"p_{k}": float(scores[k]) for k in EMO})
        records.append(rec)
    buf_frames.clear(); buf_times.clear()

while True:
    ok, frame = cap.read()
    if not ok: break
    idx += 1
    if idx % step != 0:
        continue
    t = idx / fps

    buf_frames.append(frame)
    buf_times.append(t)
    if len(buf_frames) >= MAX_FRAMES_PER_CALL:
        flush_buffer()

    pbar.update(1)

cap.release()
flush_buffer()
pbar.close()

# Save per-frame CSV (same file pattern as before)
os.makedirs(f"{BASE_DIR}/outputs", exist_ok=True)
frame_csv = f"{BASE_DIR}/outputs/{VIDEO_ID}_frame_emotions.csv"
df = pd.DataFrame(records)
df.to_csv(frame_csv, index=False)
print("Per-frame emotions CSV:", frame_csv)

# Aggregate summary (your original logic)
emo_cols = [c for c in df.columns if c.startswith("p_")]
summary = {}
if len(df):
    mode_series = df.loc[df["top_emotion"]!="no_face","top_emotion"]  # 'no_face' not used here but keep logic
    dominant = mode_series.mode().iloc[0] if len(mode_series) else "neutral"
    mean_probs = df.loc[:, emo_cols].mean().to_dict() if len(df) else {}
    summary = {
        "video_id": VIDEO_ID,
        "duration_sec": duration,
        "frames_sampled": int(len(df)),
        "dominant_emotion": dominant,
        "mean_emotions": mean_probs
    }

summary_json = f"{BASE_DIR}/outputs/{VIDEO_ID}_video_emotion_summary.json"
with open(summary_json, "w") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print("Video summary JSON:", summary_json)
summary


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
  warnings.warn(



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Qwen emotion frames:   1%|          | 2/199 [29:37<48:38:36, 888.92s/it]WARNING:py.warnings:/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



KeyboardInterrupt: 